In [ ]:
import pandas as pd
import scanpy as sc

In [ ]:
adata = sc.read_h5ad("../preprocess/adata_cd8_integrated.h5ad") # it is just cd8.

In [ ]:
DATA_DIR = "/home/roger/protocol/cancer-pseudotime-grn-workflow/scripts/data/GSE123813"

tcr_info = pd.read_csv(f"{DATA_DIR}/GSE123813_scc_tcr.txt.gz", sep="\t", index_col=0)

In [ ]:
tcr_info

In [ ]:
import pandas as pd

def extract_chains(seq_string, chain="TRA"):
    if pd.isna(seq_string):
        return []
    
    parts = str(seq_string).split(";")
    vals = [p.split(":", 1)[1] for p in parts if p.startswith(f"{chain}:")]
    return vals

# extract lists first
tcr_info["TRA_nt_list"] = tcr_info["cdr3s_nt"].apply(lambda x: extract_chains(x, "TRA"))
tcr_info["TRB_nt_list"] = tcr_info["cdr3s_nt"].apply(lambda x: extract_chains(x, "TRB"))
tcr_info["TRA_aa_list"] = tcr_info["cdr3s_aa"].apply(lambda x: extract_chains(x, "TRA"))
tcr_info["TRB_aa_list"] = tcr_info["cdr3s_aa"].apply(lambda x: extract_chains(x, "TRB"))

# find maximum number of chains
max_tra_nt = tcr_info["TRA_nt_list"].apply(len).max()
max_trb_nt = tcr_info["TRB_nt_list"].apply(len).max()
max_tra_aa = tcr_info["TRA_aa_list"].apply(len).max()
max_trb_aa = tcr_info["TRB_aa_list"].apply(len).max()

# expand TRA nt
for i in range(max_tra_nt):
    tcr_info[f"TRA{i+1}_nt"] = tcr_info["TRA_nt_list"].apply(
        lambda x: x[i] if len(x) > i else None
    )

# expand TRB nt
for i in range(max_trb_nt):
    tcr_info[f"TRB{i+1}_nt"] = tcr_info["TRB_nt_list"].apply(
        lambda x: x[i] if len(x) > i else None
    )

# expand TRA aa
for i in range(max_tra_aa):
    tcr_info[f"TRA{i+1}_aa"] = tcr_info["TRA_aa_list"].apply(
        lambda x: x[i] if len(x) > i else None
    )

# expand TRB aa
for i in range(max_trb_aa):
    tcr_info[f"TRB{i+1}_aa"] = tcr_info["TRB_aa_list"].apply(
        lambda x: x[i] if len(x) > i else None
    )

In [ ]:
common = adata.obs_names.intersection(tcr_info.index)

In [ ]:
tcr_info_aligned = tcr_info.reindex(adata.obs_names)
adata.obs = adata.obs.join(tcr_info_aligned)

In [ ]:
adata.obs

In [ ]:
tra_cols = [c for c in adata.obs.columns if c.startswith("TRA") and c.endswith("_aa")]
trb_cols = [c for c in adata.obs.columns if c.startswith("TRB") and c.endswith("_aa")]

adata.obs["TRA_set"] = adata.obs[tra_cols].apply(lambda x: set(x.dropna()), axis=1)
adata.obs["TRB_set"] = adata.obs[trb_cols].apply(lambda x: set(x.dropna()), axis=1)

In [ ]:
def has_overlap(row1, row2):
    return (
        len(row1["TRA_set"] & row2["TRA_set"]) > 0 or
        len(row1["TRB_set"] & row2["TRB_set"]) > 0
    )

In [ ]:
# combine all chains into one set
adata.obs["TCR_set"] = adata.obs.apply(
    lambda r: r["TRA_set"] | r["TRB_set"], axis=1
)

# explode → one row per chain
df = adata.obs["TCR_set"].explode().reset_index()
df.columns = ["cell", "chain"]

# cells sharing same chain
matches_df = df.merge(df, on="chain")
matches_df = matches_df[matches_df["cell_x"] != matches_df["cell_y"]]

matches_df = matches_df[["cell_x", "cell_y"]].drop_duplicates()

In [ ]:
match_counts = matches_df["cell_x"].value_counts()
adata.obs["n_matches"] = adata.obs.index.map(match_counts).fillna(0)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

table = pd.crosstab(
    adata.obs["TRB1_aa"],  # TRA2 yes
    [adata.obs["celltype"], adata.obs["treatment"]]
)

# top clones
top = table.sum(axis=1).sort_values(ascending=False).head(30).index

sns.heatmap(table.loc[top], cmap="viridis")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

table = pd.crosstab(
    adata.obs["TRA1_aa"],  # TRA2 yes
    [adata.obs["celltype"], adata.obs["treatment"]]
)

# top clones
top = table.sum(axis=1).sort_values(ascending=False).head(15).index

sns.heatmap(table.loc[top], cmap="viridis")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
import pandas as pd

# Make sure values are numeric
trb_counts = table.apply(pd.to_numeric, errors="coerce").fillna(0)

# Number of celltype/treatment columns where each TRB appears
n_columns_present = (trb_counts > 0).sum(axis=1)

# Total number of cells per TRB
n_total_cells = trb_counts.sum(axis=1)

# Create summary table
trb_summary = pd.DataFrame({
    "n_columns_present": n_columns_present,
    "n_total_cells": n_total_cells
})

# Sort by broadest presence first, then most cells
trb_summary = trb_summary.sort_values(
    ["n_columns_present", "n_total_cells"],
    ascending=False
)

trb_summary.sort_values(['n_total_cells'], ascending=False).head(20)

In [ ]:
import pandas as pd

# Make sure values are numeric
trb_counts = table.apply(pd.to_numeric, errors="coerce").fillna(0)

# Number of celltype/treatment columns where each TRB appears
n_columns_present = (trb_counts > 0).sum(axis=1)

# Total number of cells per TRB
n_total_cells = trb_counts.sum(axis=1)

# Create summary table
trb_summary = pd.DataFrame({
    "n_columns_present": n_columns_present,
    "n_total_cells": n_total_cells
})

# Sort by broadest presence first, then most cells
trb_summary = trb_summary.sort_values(
    ["n_columns_present", "n_total_cells"],
    ascending=False
)

trb_summary.sort_values(['n_total_cells'], ascending=False).head(20)

In [ ]:
# so we already know that the trajectories are different, so it makes sense to look at it before and after, and we can track a clonotype
# from mem-post to ex-post so it means that it probably is a trajectory, so it matches the outcome...

# we need to comprovate everything and see if this makes some kind of sense or what?

In [ ]:
# it seems that naive pre and mem_post also share a lot of clonotypes, this means that we could suggest that they can be the "naive" population...

In [ ]:
# what if mem post have come from naive pre and this could be amazing...

In [ ]:
# can we map the clonotypes on the umap, then would be nice to see it. 

In [ ]:
import scanpy as sc

for clonotype in ["TRA1_aa", "TRB1_aa"]:
    top10 = adata.obs[clonotype].astype(str).value_counts().head(10).index

    adata.obs[f"top10_{clonotype}"] = adata.obs[clonotype].astype(str).where(
        adata.obs[clonotype].astype(str).isin(top10),
        other="other"
    ).astype("category")

sc.pl.umap(
    adata,
    color=["top10_TRA1_aa", "top10_TRB1_aa", "celltype", "treatment"]
)

In [ ]:
adata.obs['TRB_set']

In [ ]:


target_trb = "CAPQSAGNKLTF" # es com el da dalt, mem to act...
target_trb = "CAVRDAGKLIF"
target_trb = "CALYAGGTSYGKLTF"

adata.obs[f"TRB_{target_trb}"] = adata.obs["TRA_set"].apply(
    lambda x: target_trb in x if isinstance(x, set) else target_trb in str(x)
)

adata.obs[f"TRB_{target_trb}"].value_counts()

sc.pl.umap(
    adata,
    color=[f"TRB_{target_trb}"],
    groups=[True],
    size=25
)

In [ ]:
# veig que n'hi ha molts de Naive Pre, i Mem Post, per tant, també es bo que settegem la root a memory quan es post.
# i podem confirmar-ho a partir d'algun TCR interessant...

CAPQSAGNKLTF 	
# aquest es espectacular la veritat!!!
target_trb = "CASSLGLVQPQHF"
target_trb = "CASSLGTVNTEAFF"  #tambe bo naive to mem
target_trb = "CASSQASGRVGGTDTQYF" # mem i se'n va cap a exhausted, pot ser bo.
target_trb = "CASSLGASGAYEQYF" # aquest es bo també, d'act se'n va cap a exhausted
target_trb = "CASSLPSGGSRSTDTQYF"  # de mem se'n va cap a activated, això brutal!!

adata.obs[f"TRB_{target_trb}"] = adata.obs["TRB_set"].apply(
    lambda x: target_trb in x if isinstance(x, set) else target_trb in str(x)
)

adata.obs[f"TRB_{target_trb}"].value_counts()

sc.pl.umap(
    adata,
    color=[f"TRB_{target_trb}"],
    groups=[True],
    size=25
)

In [ ]:
target_trb = "CASRRILAGGPEGTQYF"

adata.obs[f"TRB_{target_trb}"] = adata.obs["TRB_set"].apply(
    lambda x: target_trb in x if isinstance(x, set) else target_trb in str(x)
)

adata.obs[f"TRB_{target_trb}"].value_counts()

In [ ]:
sc.pl.umap(adata, color = ['celltype'] )

In [ ]:
sc.pl.umap(
    adata,
    color=[f"TRB_{target_trb}"],
    groups=[True],
    size=25
)

In [ ]:
sc.pl.umap(adata, color=[{"CASRRILAGGPEGTQYF"}])

In [ ]:
top